# Recall Decoder

A RAG project on NHTSA car recall data. I clean the data, make the recalls searchable (meaning search + keyword search), use an LLM (Groq) to answer questions with recall numbers as proof, and test how often it's right.

## Step 1: Load and clean the data

The file has no header row. RCL.txt lists all 29 columns, so I keep only the ones I need.

In [15]:
import csv
import pandas as pd

# the file has no header row, so I read everything as text first
# QUOTE_NONE: many rows have " marks inside the text, so pandas should treat them as normal characters
df = pd.read_csv("data/raw/FLAT_RCL_POST_2010.zip", sep="\t", header=None, dtype=str, quoting=csv.QUOTE_NONE)
df.shape

(246899, 29)

In [16]:
# keep only the columns I need (the numbers come from RCL.txt, counting from 0)
df = df[[1, 2, 3, 4, 6, 10, 15, 19, 20, 21, 27, 28]]
df.columns = ["recall_id", "make", "model", "year", "component", "type", "report_date",
              "summary", "consequence", "remedy", "do_not_drive", "park_outside"]
df.head()

,recall_id,make,model,year,component,type,report_date,summary,consequence,remedy,do_not_drive,park_outside
0,10V143000,GLAVAL,TITAN,2006,EQUIPMENT ADAPTIVE/MOBILITY,V,20100412,GLAVAL BUS IS RECALLING CERTAIN MODEL YEAR 200...,THE USER OF THE LIFT COULD BE INJURED SHOULD T...,GLAVAL BUS IS WORKING WITH RICON AND RICON WIL...,No,No
1,10V080000,GEM,eS,2009,PARKING BRAKE,V,20100304,GEM IS RECALLING CERTAIN MODEL YEAR 2009 NEIGH...,THE VEHICLE COULD ROLL AWAY ON A HILL CREATING...,DEALERS WILL REPLACE THE PARKING BRAKE LEVER A...,No,No
2,10V419000,NU WA,DISCOVERY SERIES,2010,EQUIPMENT,V,20100921,NU WA IS RECALLING CERTAIN MODEL YEAR 2000-201...,AN OVERHEATED RECEIVER COULD CAUSE A FIRE.,NU WA IS WORKING WITH DIMPLEX AND DIMPLEX WILL...,No,No
3,10V044000,KEYSTONE,FUZION,2010,STRUCTURE:BODY:DOOR,V,20100218,KEYSTONE IS RECALLING CERTAIN MODEL YEAR 2010 ...,A FALLING DOOR CAN LEAD TO AN INCREASED RISK O...,DEALERS WILL REPLACE THE RAMP DOOR CABLE HARDW...,No,No
4,10V082000,NOVA BUS,NOVA LFS,2009,"FUEL SYSTEM, OTHER:DELIVERY:FUEL PUMP",V,20100305,NOVA BUS IS RECALLING CERTAIN MODEL YEAR 2009-...,THE MANUFACTURER HAS NOT YET PROVIDED A CONSEQ...,NOVA BUS IS WORKING WITH CUMMINS AND CUMMINS I...,No,No


In [17]:
# keep vehicle recalls (type V) for model years 2011 to 2026
df = df[df["type"] == "V"]
df["year"] = pd.to_numeric(df["year"])
df = df[(df["year"] >= 2011) & (df["year"] <= 2026)]
print(df.shape)
print(df["recall_id"].nunique(), "recalls")

(189274, 12)
12512 recalls


In [18]:
# clean the names and the text
df["make"] = df["make"].str.strip().str.upper()
df["model"] = df["model"].str.strip().str.upper()
for col in ["summary", "consequence", "remedy"]:
    df[col] = df[col].str.replace(r"\s+", " ", regex=True).str.strip()

# part = the text before the first ":" in the component, like "AIR BAGS"
df["part"] = df["component"].str.split(":").str[0].str.title()

# turn the date and the Yes/No columns into real types
df["report_date"] = pd.to_datetime(df["report_date"], format="%Y%m%d")
df["do_not_drive"] = df["do_not_drive"] == "Yes"
df["park_outside"] = df["park_outside"] == "Yes"
df.head()

,recall_id,make,model,year,component,type,report_date,summary,consequence,remedy,do_not_drive,park_outside,part
73,10V103000,IC BUS,FESB,2011,"SERVICE BRAKES, HYDRAULIC",V,2010-03-16,NAVISTAR HAS NOTIFIED NHTSA ABOUT A DEFECT IN ...,"RELEASE OF THE PARKING BRAKE MAY CAUSE SUDDEN,...",NAVISTAR WILL NOTIFY OWNERS AND WILL REPAIR TH...,False,False,"Service Brakes, Hydraulic"
84,10V103000,IC BUS,BE,2011,"SERVICE BRAKES, HYDRAULIC",V,2010-03-16,NAVISTAR HAS NOTIFIED NHTSA ABOUT A DEFECT IN ...,"RELEASE OF THE PARKING BRAKE MAY CAUSE SUDDEN,...",NAVISTAR WILL NOTIFY OWNERS AND WILL REPAIR TH...,False,False,"Service Brakes, Hydraulic"
85,10V103000,IC BUS,CE,2011,"SERVICE BRAKES, HYDRAULIC",V,2010-03-16,NAVISTAR HAS NOTIFIED NHTSA ABOUT A DEFECT IN ...,"RELEASE OF THE PARKING BRAKE MAY CAUSE SUDDEN,...",NAVISTAR WILL NOTIFY OWNERS AND WILL REPAIR TH...,False,False,"Service Brakes, Hydraulic"
105,10V124000,HEARTLAND,NORTH COUNTRY,2011,EQUIPMENT:OTHER:LABELS,V,2010-03-30,HEARTLAND HAS NOTIFIED NHTSA OF A NONCOMPLIANC...,TOWING A VEHICLE WITH THE AXLES LOADED BEYOND ...,DEALERS WILL MAIL TO CONSUMERS THE CORRECTED L...,False,False,Equipment
108,10V124000,HEARTLAND,TRAIL RUNNER,2011,EQUIPMENT:OTHER:LABELS,V,2010-03-30,HEARTLAND HAS NOTIFIED NHTSA OF A NONCOMPLIANC...,TOWING A VEHICLE WITH THE AXLES LOADED BEYOND ...,DEALERS WILL MAIL TO CONSUMERS THE CORRECTED L...,False,False,Equipment


In [19]:
# table 1: one row per recall (the text is the same on every row of a recall)
recalls = df.drop_duplicates("recall_id")
recalls = recalls[["recall_id", "report_date", "part", "summary", "consequence", "remedy", "do_not_drive", "park_outside"]]

# table 2: which cars each recall covers
cars = df[["recall_id", "make", "model", "year"]].drop_duplicates()

print(len(recalls), "recalls and", len(cars), "recall-car rows")
recalls.to_parquet("data/clean/recalls.parquet", index=False)
cars.to_parquet("data/clean/cars.parquet", index=False)

12512 recalls and 70766 recall-car rows


# Step 2: Graphs

In [20]:
import plotly.express as px

recalls_per_year = recalls["report_date"].dt.year.value_counts().sort_index()
px.bar(x=recalls_per_year.index, y=recalls_per_year.values, title="Recalls per year",
       labels={"x": "Year", "y": "Number of recalls"}, template="simple_white")

In [21]:
top_parts = recalls["part"].value_counts().head(10).sort_values()
px.bar(x=top_parts.values, y=top_parts.index, orientation="h", title="Top 10 parts with the most recalls",
       labels={"x": "Number of recalls", "y": ""}, template="simple_white")

In [22]:
# how many recalls each year came with a "do not drive" or "park outside" warning
warnings_per_year = recalls.groupby(recalls["report_date"].dt.year)[["do_not_drive", "park_outside"]].sum()
px.bar(warnings_per_year, barmode="group", title="Recalls with a safety warning, per year",
       labels={"report_date": "Year", "value": "Number of recalls", "variable": "Warning"}, template="simple_white")

## Step 3: Make the recalls searchable

I use two kinds of search and combine them:
- **Meaning search:** a small free model turns each recall's text into an embedding (384 numbers that capture its meaning). Chroma, a vector database, finds the recalls closest to a question.
- **Keyword search:** BM25 scores recalls by the exact words they share with the question, which helps with words like "CR-V" or "inflator".

To answer a question about one car, I search only that car's recalls.

In [23]:
import chromadb
from sentence_transformers import SentenceTransformer

# small free embedding model (it downloads once, about 130 MB)
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
client = chromadb.PersistentClient(path="data/chroma")
collection = client.get_or_create_collection("recalls")

# one piece of text per recall
texts = (recalls["part"] + ". " + recalls["summary"] + " " + recalls["consequence"] + " " + recalls["remedy"]).tolist()
ids = recalls["recall_id"].tolist()

# add them to Chroma in batches of 1000
for start in range(0, len(ids), 1000):
    batch_texts = texts[start:start + 1000]
    batch_ids = ids[start:start + 1000]
    vectors = embed_model.encode(batch_texts, normalize_embeddings=True).tolist()
    collection.upsert(ids=batch_ids, documents=batch_texts, embeddings=vectors,
                      metadatas=[{"recall_id": r} for r in batch_ids])
    print("added", start + len(batch_ids))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

added 1000
added 2000
added 3000
added 4000
added 5000
added 6000
added 7000
added 8000
added 9000
added 10000
added 11000
added 12000
added 12512


In [24]:
import re
from rank_bm25 import BM25Okapi


def tokenize(text):
    # lowercase words and numbers, keeping dashes so "cr-v" stays one word
    return re.findall(r"[a-z0-9-]+", text.lower())


bm25 = BM25Okapi([tokenize(text) for text in texts])
id_to_row = dict(zip(ids, range(len(ids))))  # recall number -> its position in texts

In [25]:
def car_recall_ids(make, model, year):
    # all the recall numbers that cover this car
    match = cars[(cars["make"] == make) & (cars["model"] == model) & (cars["year"] == year)]
    return match["recall_id"].tolist()


def search_embeddings(question, car_ids, k=5):
    # meaning search: the recalls closest to the question in Chroma
    vector = embed_model.encode([question], normalize_embeddings=True).tolist()
    result = collection.query(query_embeddings=vector, n_results=min(k, len(car_ids)),
                              where={"recall_id": {"$in": car_ids}})
    return result["ids"][0]


def search_keywords(question, car_ids, k=5):
    # keyword search: BM25 scores every recall, then I keep this car's best ones
    scores = bm25.get_scores(tokenize(question))
    ranked = sorted(car_ids, key=lambda r: scores[id_to_row[r]], reverse=True)
    return ranked[:k]


def search_hybrid(question, car_ids, k=5):
    # combine both: a recall near the top of either list gets more points
    if len(car_ids) == 0:
        return []
    points = {}
    for ranked in [search_embeddings(question, car_ids, 20), search_keywords(question, car_ids, 20)]:
        for rank, recall_id in enumerate(ranked):
            points[recall_id] = points.get(recall_id, 0) + 1 / (60 + rank)
    return sorted(points, key=points.get, reverse=True)[:k]

In [26]:
car_ids = car_recall_ids("HONDA", "CR-V", 2019)
question = "Is there a problem with the air bags?"
print(len(car_ids), "recalls for this car")
print("Embeddings:", search_embeddings(question, car_ids))
print("Keywords:  ", search_keywords(question, car_ids))
print("Hybrid:    ", search_hybrid(question, car_ids))

8 recalls for this car
Embeddings: ['26V332000', '19V383000', '19V694000', '23V158000', '23V858000']
Keywords:   ['19V383000', '26V332000', '19V694000', '23V158000', '19V865000']
Hybrid:     ['26V332000', '19V383000', '19V694000', '23V158000', '23V858000']


## Step 4: Answer questions with an LLM (RAG)

Groq runs the LLM. For each question, I find the top 5 recalls for the car (hybrid search) and give only those to the LLM. It has to cite the recall numbers it used, and say "I don't know" when the recalls don't cover the question.

In [28]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()  # reads GROQ_API_KEY from the .env file
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

BIG_MODEL = "openai/gpt-oss-120b"
SMALL_MODEL = "openai/gpt-oss-20b"


def ask_llm(prompt, llm=BIG_MODEL):
    response = groq_client.chat.completions.create(
        model=llm, messages=[{"role": "user", "content": prompt}], temperature=0)
    return response.choices[0].message.content


print(ask_llm("In one sentence, what is a car recall?"))

A car recall is a manufacturer‑initiated request to return, repair, or replace vehicles that have been found to have safety‑related defects or non‑compliance with regulations.


In [29]:
def answer(question, make, model, year, llm=BIG_MODEL):
    # RAG: find the best recalls for this car, then the LLM answers from them only
    found = search_hybrid(question, car_recall_ids(make, model, year))
    sources = ""
    for recall_id in found:
        sources += "Recall " + recall_id + ": " + texts[id_to_row[recall_id]] + "\n\n"
    prompt = f"""You help car owners understand safety recalls.
Answer the question using ONLY the recalls below, and cite the recall numbers you used.
If the recalls don't answer the question, say "I don't know based on these recalls."

Car: {year} {make} {model}
Question: {question}

Recalls:
{sources}"""
    return ask_llm(prompt, llm), found


def explain(recall_id):
    # plain words for one recall
    prompt = "Explain this car recall to a car owner in 2 or 3 short, simple sentences:\n\n" + texts[id_to_row[recall_id]]
    return ask_llm(prompt)

In [30]:
reply, used = answer("Is there a problem with the air bags?", "HONDA", "CR-V", 2019)
print(reply)
print("Recalls used:", used)
print()
print(explain(used[0]))

Yes. Several safety recalls for the 2019 Honda CR‑V involve the air‑bag system:

* **Recall 19V383000** – The metal core of the steering wheel can have burrs that damage the harnesses inside, which may **disable the driver’s air bag or cause it to deploy unintentionally**. Dealers will install a protective cover and replace the clockspring and harnesses【19V383000】.  

* **Recall 26V332000** – The front‑passenger seat weight sensor may crack and short‑circuit, which can cause the **air bags to deploy unintentionally during a crash**. Dealers will replace the sensor【26V332000】.  

* **Recall 19V694000** – The owner’s guide incorrectly describes when the “Passenger Airbag Off” indicator should illuminate, which could lead to **misunderstanding of the passenger‑air‑bag operation**. Honda will provide a corrective sticker【19V694000】.

These recalls indicate that the 2019 CR‑V does have known air‑bag‑related issues that Honda has addressed with dealer repairs or information updates.
Recalls 

## Step 5: Look up a car by its VIN

NHTSA has a free VIN decoder. It turns a VIN into the make, model, and year, so I can find the car's recalls.

In [31]:
import requests


def decode_vin(vin):
    # NHTSA's free VIN decoder turns a VIN into make, model, and year
    url = "https://vpic.nhtsa.dot.gov/api/vehicles/decodevinvalues/" + vin + "?format=json"
    result = requests.get(url, timeout=10).json()["Results"][0]
    return result["Make"].upper(), result["Model"].upper(), int(result["ModelYear"])


make, model, year = decode_vin("3VV3B7AXXNM")  # the first 11 characters of a VIN are enough
print(year, make, model, "->", len(car_recall_ids(make, model, year)), "recalls")

2022 VOLKSWAGEN TIGUAN -> 5 recalls


## Step 6: Test how well it works

1. Make a sheet of 25 real recalls on popular cars. I only use cars with at least 10 recalls, so finding the right one in the top 5 isn't automatic.
2. I write one question for each row, plus 5 trick questions about problems the car never had (recall_id = `none`).
3. Score the search: did each search type find the right recall in its top 5?
4. Get answers three ways: RAG with the big model, RAG with the small model, and the big model alone (no search).
5. I grade every answer by hand: **right**, **wrong**, or **made up**. Made up = the answer names a recall, problem, or fix that isn't in the real recall.

In [32]:
# 25 recalls on popular cars that have at least 10 recalls
popular = ["HONDA", "TOYOTA", "FORD", "CHEVROLET", "NISSAN", "HYUNDAI", "KIA", "JEEP", "SUBARU", "TESLA"]
counts = cars.groupby(["make", "model", "year"])["recall_id"].transform("count")
pool = cars[cars["make"].isin(popular) & (counts >= 10)]
sheet = pool.sample(25, random_state=1).merge(recalls[["recall_id", "part", "summary"]], on="recall_id")
sheet["question"] = ""
sheet.to_csv("data/test_questions.csv", index=False)
sheet.head()

,recall_id,make,model,year,part,summary,question
0,23V598000,FORD,BRONCO,2021,Back Over Prevention,Ford Motor Company (Ford) is recalling certain...,
1,25V614000,FORD,MUSTANG,2015,Seat Belts,Ford Motor Company (Ford) is recalling certain...,
2,23V598000,FORD,TRANSIT,2022,Back Over Prevention,Ford Motor Company (Ford) is recalling certain...,
3,18V739000,JEEP,CHEROKEE,2019,Power Train,Chrysler(FCA US LLC) is recalling certain 2019...,
4,15V005000,FORD,FOCUS,2014,"Fuel System, Gasoline",Ford Motor Company (Ford) notified the agency ...,


In [34]:
tests = pd.read_csv("data/test_questions.csv")
real = tests[tests["recall_id"] != "none"]

hits = {"Keywords": 0, "Embeddings": 0, "Hybrid": 0}
for i, row in real.iterrows():
    car_ids = car_recall_ids(row["make"], row["model"], row["year"])
    if row["recall_id"] in search_keywords(row["question"], car_ids):
        hits["Keywords"] += 1
    if row["recall_id"] in search_embeddings(row["question"], car_ids):
        hits["Embeddings"] += 1
    if row["recall_id"] in search_hybrid(row["question"], car_ids):
        hits["Hybrid"] += 1

search_scores = pd.DataFrame({"search": list(hits.keys()),
                              "found_in_top_5_%": [round(100 * h / len(real)) for h in hits.values()]})
search_scores.to_csv("data/search_scores.csv", index=False)
search_scores

,search,found_in_top_5_%
0,Keywords,80
1,Embeddings,76
2,Hybrid,84


In [35]:
import time

rag, small, alone = [], [], []
for i, row in tests.iterrows():
    car = str(row["year"]) + " " + row["make"] + " " + row["model"]
    rag.append(answer(row["question"], row["make"], row["model"], row["year"])[0])
    small.append(answer(row["question"], row["make"], row["model"], row["year"], SMALL_MODEL)[0])
    alone.append(ask_llm("Answer this question about the " + car + ". Include the recall number if you know it.\n\n" + row["question"]))
    print("done", i + 1, "of", len(tests))
    time.sleep(20)  # Groq's free plan limits how much text you can send per minute

tests["rag_answer"] = rag
tests["small_model_answer"] = small
tests["llm_only_answer"] = alone
tests.to_csv("data/test_answers.csv", index=False)

done 1 of 30
done 2 of 30
done 3 of 30
done 4 of 30
done 5 of 30
done 6 of 30
done 7 of 30
done 8 of 30
done 9 of 30
done 10 of 30
done 11 of 30
done 12 of 30
done 13 of 30
done 14 of 30
done 15 of 30
done 16 of 30
done 17 of 30
done 18 of 30
done 19 of 30
done 20 of 30
done 21 of 30
done 22 of 30
done 23 of 30
done 24 of 30
done 25 of 30
done 26 of 30
done 27 of 30
done 28 of 30
done 29 of 30
done 30 of 30


In [37]:
# after grading: add rag_grade, small_grade, llm_only_grade columns (right / wrong / made up)
graded = pd.read_csv("data/test_answers.csv")
answer_scores = pd.DataFrame({
    "RAG (big model)": graded["rag_grade"].value_counts(normalize=True),
    "RAG (small model)": graded["small_grade"].value_counts(normalize=True),
    "AI alone": graded["llm_only_grade"].value_counts(normalize=True),
}).fillna(0).mul(100).round()
answer_scores.to_csv("data/answer_scores.csv")
answer_scores

,RAG (big model),RAG (small model),AI alone
made up,0.0,0.0,100.0
right,93.0,90.0,0.0
wrong,7.0,10.0,0.0
